# KULLM3 · KoE5 로컬 다중 문서 RAG

PDF/HWP 로딩, 500자 청킹(중첩 100자), FAISS 저장과 BM25 Hybrid Search를 유지합니다.
LLM은 `nlpai-lab/KULLM3`, 임베딩은 `nlpai-lab/KoE5`입니다. OpenAI API 키는 필요하지 않습니다.

설치 셀 실행 후 커널을 재시작하고 순서대로 실행하세요. CUDA용 PyTorch와 충분한 NVIDIA GPU 메모리가 필요합니다. KoE5는 CPU에서 실행해 LLM용 GPU 메모리를 확보합니다. 모델 최초 다운로드에는 인터넷과 디스크 공간이 필요합니다.

KULLM3 모델 카드의 Transformers 4.38.2 안내는 2024년 기록입니다. 최신 패키지 조합의 실제 생성 성공은 아래 간단 테스트로 확인하세요. 임의로 구버전을 고정하지 않습니다.

기존 OpenAI FAISS 인덱스는 재사용하지 않습니다. 새 저장 폴더는 `./faiss_index_koe5`입니다. 접두사·정규화·토큰 제한 설정도 manifest에 보관합니다.
스캔 PDF는 OCR이 필요합니다. 실제 문서와 모델을 이용한 전체 실행은 검증하지 않았습니다.

공식 문서: https://huggingface.co/nlpai-lab/KULLM3 · https://huggingface.co/nlpai-lab/KoE5

## 첨부 파일 재현 및 수정
문제 HWP(1,073,664바이트)에서 하이퍼링크 스캐너의 읽기 위치가 고정되는 무한 루프를 재현했습니다. 경계 검사를 수정한 동일 로더로 약 0.40초에 99,932자의 텍스트 추출이 완료되었습니다. 원본 HWP와 LLM/임베딩/청킹/FAISS 설정은 유지했습니다. 이 검증은 첨부 HWP의 텍스트 추출에 한하며, 100개 전체와 모델 추론은 실행하지 않았습니다.


# 구글 드라이브 마운트

In [ ]:
# Colab에서만 Google Drive를 마운트합니다.
try:
    from google.colab import drive
except ImportError:
    print("로컬 환경: Google Drive 마운트를 생략합니다.")
else:
    drive.mount('/content/drive', force_remount=True)


# 라이브러리 설치

In [ ]:
%pip install -U langchain-text-splitters langchain-community langchain-huggingface faiss-cpu pypdf rank-bm25 langchain-hwp-hwpx-loader==0.1.1 sentence-transformers transformers accelerate bitsandbytes hwp-hwpx-parser==1.0.0


# 로컬 실행 환경 확인

In [ ]:
import torch
import importlib.metadata as metadata
print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
for package in ["transformers", "langchain-huggingface", "sentence-transformers", "accelerate", "bitsandbytes"]:
    print(package, metadata.version(package))
if not torch.cuda.is_available():
    raise RuntimeError("이 노트북의 KULLM3 4bit 설정에는 NVIDIA GPU와 CUDA용 PyTorch가 필요합니다.")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, VRAM {props.total_memory / 1024**3:.1f} GiB")

# ./files에서 PDF와 HWP 전체 불러오기


## HWP 파서 무한 루프 수정
`hwp-hwpx-parser==1.0.0`에서 필드 제어문자 뒤에 6바이트가 없는 경우 읽기 위치를 증가시키지 않는 버그를 수정합니다. 아래 셀을 파일 로딩 셀 전에 실행하세요. 노트북에는 수정 함수가 포함되어 있어 별도 파일 복사가 필요하지 않습니다.


In [ ]:
"""hwp-hwpx-parser 1.0.0의 잘린 필드 제어문자 무한 루프 수정.
본문/표/각주 추출 로직은 유지합니다. 원본 문서를 변경하지 않습니다.
"""
import struct
from typing import List
from hwp_hwpx_parser.hwp5 import HWP5Reader, CTRL_ID_HYPERLINK

def _extract_hyperlink_texts_from_para(self, para_data: bytes) -> List[str]:
    hyperlink_texts = []
    i = 0

    while i < len(para_data) - 1:
        code = struct.unpack_from("<H", para_data, i)[0]

        if code == 0x03:
            if i + 6 > len(para_data):
                i += 2
                continue
            if i + 6 <= len(para_data):
                ctrl_id = struct.unpack_from("<I", para_data, i + 2)[0]

                if ctrl_id == CTRL_ID_HYPERLINK:
                    text_start = i + 14
                    text_chars = []
                    j = text_start

                    while j < len(para_data) - 1:
                        c = struct.unpack_from("<H", para_data, j)[0]
                        if c == 0x04:
                            break
                        elif c == 0x03:
                            j += 2
                        elif 0x20 <= c < 0x10000:
                            text_chars.append(chr(c))
                            j += 2
                        else:
                            j += 2

                    if text_chars:
                        hyperlink_texts.append("".join(text_chars))
                    i = j
                    continue
                else:
                    i += 14
                    continue
        elif code == 0x04:
            i += 10
        elif code < 32:
            if code in (11, 12):
                i += 10
            elif 15 <= code <= 23:
                i += 14
            else:
                i += 2
        else:
            i += 2

    return hyperlink_texts


def apply_hwp_parser_fix():
    HWP5Reader._extract_hyperlink_texts_from_para = _extract_hyperlink_texts_from_para

apply_hwp_parser_fix()
print("HWP 파서의 제어문자 반복문 수정 적용 완료")


In [ ]:
from pathlib import Path
import json
from langchain_community.document_loaders import PyPDFLoader
from langchain_hwp_hwpx import HwpHwpxLoader
from hwp_hwpx_parser import ExtractOptions, TableStyle

FILES_DIR = Path("./files").resolve()
INDEX_DIR = Path("./faiss_index_koe5").resolve()
EXPECTED_FILE_COUNT = 100  # 확인용이며 파일 수를 제한하지 않습니다.
STOP_ON_LOAD_ERROR = True

print("현재 작업 폴더:", Path.cwd())
print("문서 폴더:", FILES_DIR)
if not FILES_DIR.is_dir():
    raise FileNotFoundError(f"문서 폴더가 없습니다: {FILES_DIR}")
file_paths = sorted(
    (p for p in FILES_DIR.rglob("*")
     if p.is_file() and p.suffix.lower() in {".pdf", ".hwp"}),
    key=lambda p: str(p).lower(),
)
if not file_paths:
    raise ValueError("./files에서 PDF 또는 HWP 파일을 찾지 못했습니다.")
print(f"발견한 파일: {len(file_paths)}개")
if len(file_paths) != EXPECTED_FILE_COUNT:
    print(f"참고: 예상 {EXPECTED_FILE_COUNT}개와 다릅니다. 발견한 파일 전체를 처리합니다.")

pages = []  # PDF 페이지와 HWP 문서를 담는 기존 RAG 입력 변수
loaded_files = []
load_errors = []
empty_pdf_pages = []
for number, file_path in enumerate(file_paths, start=1):
    try:
        if file_path.suffix.lower() == ".pdf":
            loader = PyPDFLoader(str(file_path))
        else:
            loader = HwpHwpxLoader(
                file_path=file_path,
                mode="single",
                extract_options=ExtractOptions(table_style=TableStyle.MARKDOWN),
                include_tables=True,
                include_notes=True,
                on_encrypted="raise",
                on_invalid="raise",
                on_error="raise",
            )
        import time
        load_started = time.perf_counter()
        print(f"[{number}/{len(file_paths)}] 추출 시작: {file_path.name}", flush=True)
        documents = loader.load()
        print(f"추출 완료: {time.perf_counter() - load_started:.2f}초", flush=True)
        valid_documents = []
        for doc in documents:
            if not doc.page_content.strip():
                if file_path.suffix.lower() == ".pdf":
                    empty_pdf_pages.append({
                        "source": str(file_path), "page": doc.metadata.get("page")
                    })
                continue
            doc.metadata.update({
                "source": str(file_path),
                "file_name": file_path.name,
                "relative_path": file_path.relative_to(FILES_DIR).as_posix(),
                "file_type": file_path.suffix.lower().lstrip("."),
            })
            valid_documents.append(doc)
        if not valid_documents:
            raise ValueError("추출된 텍스트가 없습니다. 스캔/OCR 또는 문서 형식을 확인하세요.")
        pages.extend(valid_documents)
        loaded_files.append(str(file_path))
        print(f"[{number}/{len(file_paths)}] 성공: {file_path.name} ({len(valid_documents)}개 문서 단위)")
    except Exception as exc:
        load_errors.append({"source": str(file_path), "error": f"{type(exc).__name__}: {exc}"})
        print(f"[{number}/{len(file_paths)}] 실패: {file_path.name} - {exc}")

print(f"파일 로딩: 성공 {len(loaded_files)}개 / 실패 {len(load_errors)}개")
print(f"텍스트가 없는 PDF 페이지: {len(empty_pdf_pages)}개 (OCR 필요 여부 확인)")
Path("./document_load_report.json").write_text(
    json.dumps({"loaded_files": loaded_files, "load_errors": load_errors,
                "empty_pdf_pages": empty_pdf_pages}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
if load_errors and STOP_ON_LOAD_ERROR:
    raise RuntimeError("읽기 실패 파일이 있습니다. document_load_report.json을 확인하고 재실행하세요.")
if not pages:
    raise ValueError("청킹할 문서 텍스트가 없습니다.")

PDF는 페이지별로, HWP는 파일 단위로 읽어 `pages`에 합칩니다. 빈 PDF 페이지는 별도로 보고합니다. PDF `page`는 0부터 시작하며 HWP에는 실제 페이지 번호를 부여하지 않습니다.


# 문서 청킹

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    add_start_index=True,
)
all_splits = [doc for doc in text_splitter.split_documents(pages)
              if doc.page_content.strip()]
for chunk_id, doc in enumerate(all_splits):
    doc.metadata["chunk_id"] = chunk_id
if not all_splits:
    raise ValueError("생성된 청크가 없습니다.")
print("전체 청크 수:", len(all_splits))
print("청킹된 파일 수:", len({doc.metadata["source"] for doc in all_splits}))


청크 크기는 500자, 중첩은 100자입니다. 파일/페이지 메타데이터와 `start_index`, `chunk_id`를 보존합니다.


# FAISS 구축 및 로컬 저장


In [ ]:
from datetime import datetime, timezone
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer
import gc
import torch

EMBEDDING_MODEL = "nlpai-lab/KoE5"
BATCH_SIZE = 64
EMBEDDING_CONFIG = {
    "model": EMBEDDING_MODEL,
    "dimension": 1024,
    "document_prefix": "passage: ",
    "query_prefix": "query: ",
    "normalize_embeddings": True,
    "max_seq_length": 512,
}

class KoE5Embeddings(Embeddings):
    """원문은 유지하고, 임베딩할 때만 E5 접두사를 추가합니다."""
    def __init__(self):
        self.encoder = SentenceTransformer(EMBEDDING_MODEL, device="cuda")
        self.encoder.max_seq_length = EMBEDDING_CONFIG["max_seq_length"]
        self.truncated_document_count = 0

    def embed_documents(self, texts):
        prefixed = [EMBEDDING_CONFIG["document_prefix"] + text for text in texts]
        lengths = self.encoder.tokenizer(prefixed, truncation=False)["input_ids"]
        count = sum(len(ids) > self.encoder.max_seq_length for ids in lengths)
        self.truncated_document_count += count
        if count:
            print(f"주의: 이번 배치 {count}개 청크가 512토큰을 초과하여 임베딩 시 잘립니다. 토큰 기반 청킹을 검토하세요.")
        return self.encoder.encode(prefixed, batch_size=BATCH_SIZE,
            normalize_embeddings=True, show_progress_bar=False).tolist()

    def embed_query(self, text):
        prefixed = EMBEDDING_CONFIG["query_prefix"] + text
        if len(self.encoder.tokenizer(prefixed)["input_ids"]) > self.encoder.max_seq_length:
            raise ValueError("질문이 KoE5의 512토큰 제한을 초과합니다.")
        return self.encoder.encode([prefixed], normalize_embeddings=True,
            show_progress_bar=False)[0].tolist()

embeddings = KoE5Embeddings()
if len(embeddings.embed_query("임베딩 동작 확인")) != 1024:
    raise ValueError("KoE5 임베딩 차원이 예상과 다릅니다.")

# 첫 배치의 실제 임베딩 차원에 맞춰 FAISS 인덱스를 생성합니다.
vector_store = None
for start in range(0, len(all_splits), BATCH_SIZE):
    batch = all_splits[start:start + BATCH_SIZE]
    ids = [str(doc.metadata["chunk_id"]) for doc in batch]
    if vector_store is None:
        vector_store = FAISS.from_documents(batch, embeddings, ids=ids)
    else:
        vector_store.add_documents(batch, ids=ids)
    print(f"임베딩 및 FAISS 저장: {min(start + BATCH_SIZE, len(all_splits))}/{len(all_splits)} 청크")

# index.faiss: 벡터 인덱스, index.pkl: 문서/메타데이터와 ID 매핑
INDEX_DIR.mkdir(parents=True, exist_ok=True)
vector_store.save_local(str(INDEX_DIR))
# BM25 재구축에도 사용할 수 있도록 청크를 JSON으로 저장합니다.
(INDEX_DIR / "chunks.json").write_text(
    json.dumps([{"page_content": doc.page_content, "metadata": doc.metadata}
                for doc in all_splits], ensure_ascii=False, indent=2),
    encoding="utf-8",
)
manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "files_dir": str(FILES_DIR),
    "embedding_model": EMBEDDING_MODEL,
    "embedding_config": EMBEDDING_CONFIG,
    "truncated_document_count": embeddings.truncated_document_count,
    "embedding_dimension": vector_store.index.d,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "file_count_found": len(file_paths),
    "file_count_loaded": len(loaded_files),
    "chunk_count": len(all_splits),
    "loaded_files": loaded_files,
    "load_errors": load_errors,
    "empty_pdf_pages": empty_pdf_pages,
}
(INDEX_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
print("FAISS 로컬 저장 완료:", INDEX_DIR)
print("벡터 개수:", vector_store.index.ntotal)

embeddings.encoder.to("cpu")
gc.collect()
torch.cuda.empty_cache()


저장 위치는 `./faiss_index`입니다. `index.faiss`, `index.pkl`, `chunks.json`, `manifest.json`을 함께 보관하세요. 이후 RAG 및 Hybrid Search 셀은 전체 파일의 청크를 검색합니다.


# KULLM3 4bit 로딩

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.runnables import RunnableLambda

language_model_name = "nlpai-lab/KULLM3"
if not torch.cuda.is_available():
    raise RuntimeError("KULLM3 4bit 실행에는 CUDA용 PyTorch와 NVIDIA GPU가 필요합니다.")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(language_model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    language_model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
#print("LLM 장치 배치:", model.hf_device_map)

In [ ]:
model.config.use_cache = True
MAX_NEW_TOKENS = 512
llm_pipeline = pipeline(
    "text-generation", model=model, tokenizer=tokenizer,
    do_sample=False, repetition_penalty=1.2,
    return_full_text=False, max_new_tokens=MAX_NEW_TOKENS,
    use_cache=True, pad_token_id=tokenizer.pad_token_id,
)
llm = HuggingFacePipeline(pipeline=llm_pipeline, model_id=language_model_name)

# 문자열 PromptTemplate에도 모델 고유의 채팅 템플릿을 반드시 적용합니다.
def render_chat_prompt(value):
    text = value.to_string() if hasattr(value, "to_string") else str(value)
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": text}],
        tokenize=False, add_generation_prompt=True,
    )

def generate_chat(value):
    rendered = render_chat_prompt(value)
    input_tokens = len(tokenizer(rendered, add_special_tokens=False)["input_ids"])
    limit = model.config.max_position_embeddings
    if input_tokens + MAX_NEW_TOKENS > limit:
        raise ValueError(f"입력 {input_tokens} + 출력 {MAX_NEW_TOKENS}토큰이 모델 한도 {limit}를 초과합니다. 검색 청크 수를 줄이세요.")
    # 채팅 템플릿에 포함된 특수 토큰을 중복 추가하지 않습니다.
    return llm.invoke(rendered, pipeline_kwargs={"add_special_tokens": False})

chat_model = RunnableLambda(generate_chat)

KULLM3 채팅 템플릿을 적용하고 입력과 출력의 전체 토큰 수를 검사합니다. `chat_model.invoke()`는 문자열을 반환합니다.

# Retriever, LLM 테스트: 생략 가능


In [ ]:
import time

question = "연말 정산 때 비거주자가 주의할 점을 알려 줘."

start = time.time()

docs = retriever.invoke(question)

print(f"검색 시간: {time.time() - start:.2f}초")
print("검색된 chunk:", len(docs))

In [ ]:
test_prompt = """
질문: 대한민국의 수도는 어디인가요?
한국어로 짧게 답변하세요.
"""

start = time.time()

result = chat_model.invoke(test_prompt)

print(f"LLM 생성 시간: {time.time() - start:.2f}초")
print(result)

In [ ]:
question = "연말 정산 때 비거주자가 주의할 점을 알려 줘."
docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)
print("검색된 chunk:", len(docs))
print("context 글자 수:", len(context))
print("KULLM3 context token 수:", len(tokenizer(context, add_special_tokens=False)["input_ids"]))
# 실제 전체 프롬프트의 토큰 제한은 generate_chat에서 검사합니다.

# RAG

In [ ]:
from langchain_core.prompts import PromptTemplate


template = """
아래에 주어진 맥락을 이용해 질문에 대해 답변해 줘.
주어진 맥락으로 답변이 어려운 상황이라면, 그냥 모른다고 답하면 되고 억지로 답변을 꾸며 내지 마.
최대한 자세하게 답변해 줘.
반드시 한국어로 답변해야 해.

맥락:
{context}

질문:
{question}
"""

prompt = PromptTemplate.from_template(template)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    print("검색된 문서 수:", len(docs))
    print(docs)
    return "\n\n".join(doc.page_content for doc in docs)

retrieval_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

In [ ]:
question = "복수의결권 및 스톡옵션 시스템 고도화는 어떻게 추진돼?."

result = retrieval_chain.invoke(question)
print(result)

In [ ]:
question = "고려대학교 차세대 포털·학사 정보시스템 구축 사업의 추진 배경 및 필요성에 대해 알려줘."

result = retrieval_chain.invoke(question)
print(result)

In [ ]:
question = "2024년 개정 세법 중에 월세와 관련한 내용이 있을까?"

result = retrieval_chain.invoke(question)
print(result)

In [ ]:
question = "기부금 공제 때 주의할 점은?"

result = retrieval_chain.invoke(question)
print(result)

Dense 검색은 `k=4`, LLM 출력 한도는 `max_tokens=512`입니다. 아래 셀을 실행해 새 모델의 답변을 확인하세요.


# Hybrid Search (FAISS + BM25)

기존 Dense 검색(FAISS)에 키워드 기반 Sparse 검색(BM25)을 추가함.  
두 검색 결과는 **RRF(Reciprocal Rank Fusion)** 로 결합하므로 점수 범위가 서로 달라도 안정적으로 합칠 수 있음.

In [ ]:
from collections import defaultdict
from langchain_community.retrievers import BM25Retriever
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 한국어 공백 단위 토큰화를 사용.
def tokenize_korean(text):
    return text.lower().split()


dense_k = 5
sparse_k = 5
final_k = 5
rrf_constant = 60

bm25_retriever = BM25Retriever.from_documents(
    all_splits,
    preprocess_func=tokenize_korean,
)
bm25_retriever.k = sparse_k


def _document_key(doc):
    """동일 청크를 판별하기 위한 키"""
    return (
        doc.metadata.get("source"),
        doc.metadata.get("page"),
        doc.page_content,
    )


def hybrid_search(query):
    """FAISS와 BM25 순위를 RRF로 결합해 상위 문서를 반환."""
    dense_docs = vector_store.similarity_search(query, k=dense_k)
    sparse_docs = bm25_retriever.invoke(query)

    scores = defaultdict(float)
    documents = {}

    for ranked_docs in (dense_docs, sparse_docs):
        for rank, doc in enumerate(ranked_docs, start=1):
            key = _document_key(doc)
            documents[key] = doc
            scores[key] += 1.0 / (rrf_constant + rank)

    ranked_keys = sorted(scores, key=scores.get, reverse=True)
    return [documents[key] for key in ranked_keys[:final_k]]


hybrid_retriever = RunnableLambda(hybrid_search)

hybrid_retrieval_chain = (
    {
        "context": hybrid_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | chat_model
    | StrOutputParser()
)

In [ ]:
# Hybrid Search 동작 확인
question = "2024년 개정 세법 중에 월세와 관련한 내용이 있을까?"

hybrid_docs = hybrid_search(question)
for rank, doc in enumerate(hybrid_docs, start=1):
    print(f"[{rank}] page={doc.metadata.get('page')} | {doc.page_content[:150]}...")

# 최종 RAG 답변
result = hybrid_retrieval_chain.invoke(question)
print("\n[답변]\n", result)

Hybrid Search는 KoE5 FAISS 검색과 BM25 검색을 RRF로 결합하고 KULLM3로 답변합니다.

# 다음 실행: 저장된 KoE5 FAISS로 RAG 실행 (선택)

새 커널에서 설치 셀 실행 후 아래 셀만 실행할 수 있습니다. 문서 임베딩은 다시 만들지 않지만 KoE5와 KULLM3 모델은 로드합니다.
이미 위쪽 모델을 로드했다면 이 셀은 해당 객체를 재사용합니다. 직접 생성한 신뢰할 수 있는 인덱스만 로드하세요.
Hybrid Search 재사용은 이 셀 이후 기존 프롬프트·format_docs·Hybrid Search 정의 셀을 실행하세요.

In [ ]:
from pathlib import Path
import json
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS

saved_dir = Path("./faiss_index_koe5").resolve()
saved_manifest = json.loads((saved_dir / "manifest.json").read_text(encoding="utf-8"))
if "embeddings" not in globals():
    from langchain_core.embeddings import Embeddings
    from sentence_transformers import SentenceTransformer

    EMBEDDING_MODEL = "nlpai-lab/KoE5"
    BATCH_SIZE = 64
    EMBEDDING_CONFIG = {
        "model": EMBEDDING_MODEL,
        "dimension": 1024,
        "document_prefix": "passage: ",
        "query_prefix": "query: ",
        "normalize_embeddings": True,
        "max_seq_length": 512,
    }

    class KoE5Embeddings(Embeddings):
        """원문은 유지하고, 임베딩할 때만 E5 접두사를 추가합니다."""
        def __init__(self):
            self.encoder = SentenceTransformer(EMBEDDING_MODEL, device="cpu")
            self.encoder.max_seq_length = EMBEDDING_CONFIG["max_seq_length"]
            self.truncated_document_count = 0

        def embed_documents(self, texts):
            prefixed = [EMBEDDING_CONFIG["document_prefix"] + text for text in texts]
            lengths = self.encoder.tokenizer(prefixed, truncation=False)["input_ids"]
            count = sum(len(ids) > self.encoder.max_seq_length for ids in lengths)
            self.truncated_document_count += count
            if count:
                print(f"주의: 이번 배치 {count}개 청크가 512토큰을 초과하여 임베딩 시 잘립니다. 토큰 기반 청킹을 검토하세요.")
            return self.encoder.encode(prefixed, batch_size=BATCH_SIZE,
                normalize_embeddings=True, show_progress_bar=False).tolist()

        def embed_query(self, text):
            prefixed = EMBEDDING_CONFIG["query_prefix"] + text
            if len(self.encoder.tokenizer(prefixed)["input_ids"]) > self.encoder.max_seq_length:
                raise ValueError("질문이 KoE5의 512토큰 제한을 초과합니다.")
            return self.encoder.encode([prefixed], normalize_embeddings=True,
                show_progress_bar=False)[0].tolist()

    embeddings = KoE5Embeddings()
    if len(embeddings.embed_query("임베딩 동작 확인")) != 1024:
        raise ValueError("KoE5 임베딩 차원이 예상과 다릅니다.")
if saved_manifest.get("embedding_config") != EMBEDDING_CONFIG:
    raise ValueError("저장된 설정과 현재 KoE5 설정이 다릅니다. 인덱스를 다시 구축하세요.")
if saved_manifest.get("embedding_model") != EMBEDDING_MODEL:
    raise ValueError("저장된 임베딩 모델이 KoE5가 아닙니다.")
loaded_vector_store = FAISS.load_local(
    str(saved_dir), embeddings, allow_dangerous_deserialization=True,
)
if loaded_vector_store.index.d != len(embeddings.embed_query("차원 확인")):
    raise ValueError("인덱스와 질문 임베딩 차원이 다릅니다.")
if loaded_vector_store.index.d != saved_manifest["embedding_dimension"]:
    raise ValueError("인덱스 차원이 manifest와 다릅니다.")
loaded_retriever = loaded_vector_store.as_retriever(search_kwargs={"k": 4})
loaded_all_splits = [Document(**row) for row in json.loads(
    (saved_dir / "chunks.json").read_text(encoding="utf-8"))]
if "chat_model" not in globals():
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
    from langchain_huggingface import HuggingFacePipeline
    from langchain_core.runnables import RunnableLambda

    language_model_name = "nlpai-lab/KULLM3"
    if not torch.cuda.is_available():
        raise RuntimeError("KULLM3 4bit 실행에는 CUDA용 PyTorch와 NVIDIA GPU가 필요합니다.")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(language_model_name)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        language_model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    print("LLM 장치 배치:", model.hf_device_map)
    model.config.use_cache = True
    MAX_NEW_TOKENS = 512
    llm_pipeline = pipeline(
        "text-generation", model=model, tokenizer=tokenizer,
        do_sample=False, repetition_penalty=1.2,
        return_full_text=False, max_new_tokens=MAX_NEW_TOKENS,
        use_cache=True, pad_token_id=tokenizer.pad_token_id,
    )
    llm = HuggingFacePipeline(pipeline=llm_pipeline, model_id=language_model_name)

    # 문자열 PromptTemplate에도 모델 고유의 채팅 템플릿을 반드시 적용합니다.
    def render_chat_prompt(value):
        text = value.to_string() if hasattr(value, "to_string") else str(value)
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": text}],
            tokenize=False, add_generation_prompt=True,
        )

    def generate_chat(value):
        rendered = render_chat_prompt(value)
        input_tokens = len(tokenizer(rendered, add_special_tokens=False)["input_ids"])
        limit = model.config.max_position_embeddings
        if input_tokens + MAX_NEW_TOKENS > limit:
            raise ValueError(f"입력 {input_tokens} + 출력 {MAX_NEW_TOKENS}토큰이 모델 한도 {limit}를 초과합니다. 검색 청크 수를 줄이세요.")
        # 채팅 템플릿에 포함된 특수 토큰을 중복 추가하지 않습니다.
        return llm.invoke(rendered, pipeline_kwargs={"add_special_tokens": False})

    chat_model = RunnableLambda(generate_chat)

saved_prompt = PromptTemplate.from_template(
    "아래 맥락으로 질문에 한국어로 답변하세요. 맥락에 근거가 없으면 모른다고 답하세요.\n"
    "맥락:\n{context}\n\n질문: {question}"
)
loaded_retrieval_chain = (
    {"context": loaded_retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)),
     "question": RunnablePassthrough()}
    | saved_prompt | chat_model | StrOutputParser()
)
vector_store = loaded_vector_store
all_splits = loaded_all_splits
retriever = loaded_retriever
print("로드 완료:", loaded_vector_store.index.ntotal, "개 벡터")
print(loaded_retrieval_chain.invoke("문서에서 다루는 주요 내용을 알려줘."))